In [ ]:
import sys
import os
import matplotlib.pyplot as plt

# Add the library to path if not installed via pip
sys.path.append(os.path.abspath(".."))

from dynawo_notebooks.Scripts.core.grid_manager import GridManager
from dynawo_notebooks.Scripts.core.modelica_wrapper import ModelicaWrapper
from dynawo_notebooks.Scripts.core.orchestrator import Orchestrator

In [ ]:
# Create manager instance
gm = GridManager("TestGrid")

# Define topology: Generator -> Bus1 -> Line -> Bus2 -> Load
gm.create_bus("Bus1", nominal_kv=132.0)
gm.create_bus("Bus2", nominal_kv=132.0)

# Generator at Bus1 (Slack/PV) regulating to 1.0 p.u. (132 kV)
gm.add_generator("Gen1", "Bus1", target_v_pu=1.0, rated_mw=100)

# Load at Bus2
gm.add_load("Load1", "Bus2", p_mw=80, q_mvar=30)

# Transmission line
gm.add_line("Line1", "Bus1", "Bus2", r=2.0, x=15.0)

print("Network defined successfully.")

In [ ]:
# Path to Modelica file
MO_FILE = "Models/MyBESS_static.mo"
MODEL_NAME = "MyBESS_static"

# Initialize wrapper (this compiles the model)
mw = ModelicaWrapper(MO_FILE, MODEL_NAME, libraries=None)

# Verify introspection
vars = mw.inspect_variables()
print(f"Model loaded. Variables detected: {len(vars)}")

In [ ]:
orch = Orchestrator(gm, mw)

# Execute automatic synchronization
# This logs which variables are being mapped
orch.sync_models()

In [ ]:
# Simulate 5 seconds
mw.simulate(stop_time=5.0)

# Get results for validation
time, results = mw.get_trajectory()

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(time, results)
plt.title("Voltage at Bus 2 (Should be flat)")
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")

plt.subplot(1, 2, 2)
plt.plot(time, results["Gen1.P"])
plt.title("Active Power Gen 1")
plt.xlabel("Time (s)")
plt.ylabel("Power (W)")

plt.tight_layout()
plt.show()